# conv-stride-downsample — faded example 2: Recover the stride that produced an output length

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-stride-downsample`. Running the beacon reports progress on the `CNN: Stride downsample arithmetic` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Stride downsample arithmetic` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-stride-downsample`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-stride-downsample"
DD_SUBTOPIC = "CNN: Stride downsample arithmetic"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Given an input length `L`, kernel `K`, no padding, and a known output length `H_out`, the stride satisfies `H_out = (L - K) // S + 1`. Because floor division is involved, the cleanest recovery is to scan candidate strides and return the one whose formula reproduces `H_out`. This tests reading the downsample arithmetic in reverse.

## Faded exercise 2

### Faded — recover stride from a target output length

Implement `stride_for_outlen(l_in, k, h_out)` which returns the smallest positive stride `s` such that `(l_in - k) // s + 1 == h_out` (or `None` if no stride works). Complete the body of the search loop that tests each candidate stride. The test verifies the recovered stride reproduces the target via a real `F.conv1d`.

**Fill in:** The loop body that checks whether candidate stride `s` yields `h_out` via `(l_in - k) // s + 1` and returns it if so.

In [ ]:
import torch.nn.functional as F

def stride_for_outlen(l_in, k, h_out):
    for s in range(1, l_in + 1):
        if (l_in - k) // s + 1 == h_out:
            return s
    return None


def _test():
    t.manual_seed(0)
    cases = [(20, 4, 3), (32, 5, 2), (64, 3, 2), (50, 7, 4)]
    for l_in, k, s_true in cases:
        x = t.randn(1, 2, l_in)
        w = t.randn(3, 2, k)
        h_out = F.conv1d(x, w, stride=s_true).shape[-1]
        s_rec = stride_for_outlen(l_in, k, h_out)
        assert s_rec is not None
        y = F.conv1d(x, w, stride=s_rec)
        assert y.shape[-1] == h_out, (l_in, k, s_true, s_rec, h_out, y.shape[-1])


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn.functional as F

def stride_for_outlen(l_in, k, h_out):
    for s in range(1, l_in + 1):
        if (l_in - k) // s + 1 == h_out:
            return s
    return None
```
</details>